<a href="https://colab.research.google.com/github/ourharlequin/ai_workbooks/blob/main/Homework_1_Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 1: Prompt Engineering

In this homework you will practice:
1. Experimenting with generation parameters (temperature, top_p)
2. Applying prompt engineering techniques (CoT, Few-shot) to a tricky extraction task
3. Building structured output with Pydantic models
4. Implementing Schema-Guided Reasoning patterns (Cascade, Router, Cycle)
5. Designing your own SGR pipeline from scratch

In [3]:
# Install required packages
!pip install -q mistralai google-genai openai langchain langchain-core langchain-mistralai langchain-google-genai langchain-openai python-dotenv pydantic

In [4]:
import os
import json
import math
from typing import List, Optional, Literal, Union
from pydantic import BaseModel, Field
from mistralai.client import Mistral
from google import genai
from google.genai import types
from openai import OpenAI
from google.colab import userdata

In [5]:
# Load API keys and initialize clients
mistral_key = userdata.get('MISTRAL_API_KEY')
gemini_key = userdata.get('GEMINI_API_KEY')
openrouter_key = userdata.get('OPENROUTER_API_KEY')

mistral_client = Mistral(api_key=mistral_key) if mistral_key else None
gemini_client = genai.Client(api_key=gemini_key) if gemini_key else None
openrouter_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_key) if openrouter_key else None

for name, client in [("Mistral", mistral_client), ("Gemini", gemini_client), ("OpenRouter", openrouter_client)]:
    print(f"{'✅' if client else '❌'} {name} client {'ready' if client else 'not available'}")

✅ Mistral client ready
✅ Gemini client ready
✅ OpenRouter client ready


In [6]:
# Helper functions (same as in the seminar)

def call_mistral(model_name: str, user_prompt: str, system_prompt=None,
                 temperature=None, top_p=None, max_tokens=None, **kwargs) -> str:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    params = {k: v for k, v in dict(temperature=temperature, top_p=top_p, max_tokens=max_tokens).items() if v is not None}
    response = mistral_client.chat.complete(model=model_name, messages=messages, **params, **kwargs)
    return response.choices[0].message.content

def call_gemini(model_name: str, user_prompt: str, system_prompt=None,
                temperature=None, top_p=None, max_tokens=None, **kwargs) -> str:
    config_params = {k: v for k, v in dict(temperature=temperature, top_p=top_p, max_output_tokens=max_tokens).items() if v is not None}
    config_params.update(kwargs)
    config = types.GenerateContentConfig(**config_params) if config_params else None
    contents = user_prompt if not system_prompt else [system_prompt, user_prompt]
    response = gemini_client.models.generate_content(model=model_name, contents=contents, config=config)
    return response.text

def call_openrouter(model_name: str, user_prompt: str, system_prompt=None,
                    temperature=None, top_p=None, max_tokens=None, **kwargs) -> str:
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    params = {k: v for k, v in dict(temperature=temperature, top_p=top_p, max_tokens=max_tokens).items() if v is not None}
    response = openrouter_client.chat.completions.create(model=model_name, messages=messages, **params, **kwargs)
    return response.choices[0].message.content

def parse_mistral(model_name: str, user_prompt: str, response_format, system_prompt=None):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    response = mistral_client.chat.parse(model=model_name, messages=messages, response_format=response_format)
    return response.choices[0].message.parsed

def parse_gemini(model_name: str, user_prompt: str, response_format, system_prompt=None):
    config = types.GenerateContentConfig(response_mime_type="application/json", response_schema=response_format)
    contents = user_prompt if not system_prompt else [system_prompt, user_prompt]
    response = gemini_client.models.generate_content(model=model_name, contents=contents, config=config)
    return response_format(**json.loads(response.text))

def parse_openrouter(model_name: str, user_prompt: str, response_format, system_prompt=None):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    response = openrouter_client.beta.chat.completions.parse(model=model_name, messages=messages, response_format=response_format)
    return response.choices[0].message.parsed

In [7]:
# Utility: Pretty-print any Pydantic object as a nested tree

def pretty_print(obj, indent=0, name=None):
    """Print a Pydantic object (or dict/list) as a nicely indented tree."""
    prefix = "  " * indent
    connector = "├── " if indent > 0 else ""

    if hasattr(obj, 'model_dump'):
        label = name or obj.__class__.__name__
        print(f"{prefix}{connector}{label}:")
        fields = obj.model_fields
        items = list(fields.keys())
        for i, field_name in enumerate(items):
            value = getattr(obj, field_name)
            is_last = (i == len(items) - 1)
            pretty_print(value, indent + 1, name=field_name)
    elif isinstance(obj, list):
        label = name or "list"
        if len(obj) == 0:
            print(f"{prefix}{connector}{label}: []")
        elif hasattr(obj[0], 'model_dump'):
            print(f"{prefix}{connector}{label}: ({len(obj)} items)")
            for i, item in enumerate(obj):
                pretty_print(item, indent + 1, name=f"[{i}]")
        else:
            print(f"{prefix}{connector}{label}: {obj}")
    else:
        label = name or "value"
        print(f"{prefix}{connector}{label}: {obj}")


# Utility: Compare JSON extraction results with skip-list for fuzzy fields

def compare_json(result_str, expected, skip_fields=None):
    """Compare a JSON string result against expected dict.
    Fields in skip_fields are not compared — just listed for manual review.
    """
    skip_fields = skip_fields or []

    # Parse result if it's a string
    if isinstance(result_str, str):
        # Try to extract JSON from the response (may have extra text)
        try:
            result = json.loads(result_str)
        except json.JSONDecodeError:
            # Try to find JSON in the text
            import re
            json_match = re.search(r'\{[\s\S]*\}', result_str)
            if json_match:
                result = json.loads(json_match.group())
            else:
                print("Could not parse JSON from the response.")
                return
    else:
        result = result_str

    # Flatten nested dicts for comparison
    def flatten(d, prefix=""):
        items = {}
        for k, v in d.items():
            key = f"{prefix}.{k}" if prefix else k
            if isinstance(v, dict):
                items.update(flatten(v, key))
            else:
                items[key] = v
        return items

    flat_result = flatten(result)
    flat_expected = flatten(expected)

    correct = []
    wrong = []
    skipped = []

    for key, expected_val in flat_expected.items():
        # Check if this field or any parent should be skipped
        should_skip = any(skip in key for skip in skip_fields)
        if should_skip:
            skipped.append(key)
            continue

        result_val = flat_result.get(key, "MISSING")

        # Normalize for comparison
        if isinstance(expected_val, str) and isinstance(result_val, str):
            match = expected_val.lower().strip() == result_val.lower().strip()
        else:
            match = result_val == expected_val

        if match:
            correct.append((key, expected_val))
        else:
            wrong.append((key, expected_val, result_val))

    # Print results
    print(f"=== Comparison Results ===\n")
    print(f"Correct: {len(correct)}/{len(correct) + len(wrong)}")
    if correct:
        for key, val in correct:
            print(f"  ✅ {key}: {val}")
    if wrong:
        print(f"\nWrong: {len(wrong)}")
        for key, exp, got in wrong:
            print(f"  ❌ {key}: expected {exp}, got {got}")
    if skipped:
        print(f"\nSkipped (compare manually): {len(skipped)}")
        for key in skipped:
            print(f"  ⏭️  {key}")

---
# Task 1: How Generation Parameters Shape Creative Output (1 score)

Pick your favorite movie or book. You will write a prompt that asks the LLM to generate an **alternative ending** in one paragraph (4-6 sentences). Then you'll experiment with how `temperature` and `top_p` affect the result — each parameter **in isolation**.

**What to do:**
1. Set the `my_movie_or_book` variable to your favorite movie or book title
2. Write a prompt that asks the LLM to generate an alternative ending for it. Your prompt should:
   - Reference the movie/book by name (use the variable)
   - Ask for exactly one paragraph (4-6 sentences)
   - Ask the ending to be creative but faithful to the original characters and themes
3. Run the prompt with different parameter settings and compare results

**Note:** When you only set `temperature`, the model uses its default `top_p` (typically 1.0). When you only set `top_p`, the model uses its default `temperature` (varies by provider, typically ~0.7-1.0).

In [ ]:
# TODO: Set your movie/book and write the prompt
my_movie_or_book = "Тёмная башня: Извлечение троих"  # <-- change this!

# TODO: Write your prompt here. Use the my_movie_or_book variable.
prompt = f"""
Перепиши финал этой книги с условием,что Роланд умер спасая Одетту. Сцена на станции метро. Текст на 3-4 предложения
"""

### Part A — Temperature (default top_p)

Run the prompt with 4 different temperature values. Read each output carefully and rate it.

In [ ]:
# Run with different temperature values (top_p stays at model default)
temperature_values = [0.1, 0.6, 1.0, 1.5]

for temp in temperature_values:
    print(f"=== Temperature: {temp} ===\n")

    result = call_mistral("mistral-small-latest", prompt, temperature=temp)
    # result = call_gemini("gemini-2.5-flash", prompt, temperature=temp)

    print(result)
    print("\n" + "=" * 60 + "\n")

=== Temperature: 0.1 ===

Роланд бросился к Одетте, когда поезд уже набирал скорость, и в последний миг вытолкнул её из опасной зоны. Взрыв метнулся вслед за ним, поглотив его тело, и Одетта осталась одна, сжимая в руках его окровавленную куртку. Станция озарилась вспышкой, а когда дым рассеялся, на рельсах больше никого не было — только тишина и её беззвучные слёзы.


=== Temperature: 0.6 ===

**Финал:**

Когда поезд метро, несущийся из тьмы, вот-вот должен был сбить Одетту, Роланд бросился вперед, оттолкнув её в сторону. Последним, что он увидел, было её испуганное лицо, а затем — ослепляющий свет фар. Одетта осталась жива, но Роланд навсегда остался там, в сером сумраке станции, став последней жертвой войны, которую так долго пытался остановить.


=== Temperature: 1.0 ===

Финал с заменой роли Роланда:

Когда поезд метро помчался к ограде, Роланд не отступил — он бросился перед Одеттой, будто оборонился от самой смерти. Взрыв жара смял его тело, но она осталась целой, его рука всё е

**TODO:** Fill in the table based on your observations (1 = lowest, 5 = highest):

| Temperature | Creativity (1-5) | Coherence (1-5) | Faithfulness to original (1-5) |
|------------|-----------------|-----------------|-------------------------------|
| 0.1 |3|5|3|
| 0.6 |4|5|3|
| 1.0 |5|4|5|
| 1.5 |0|0|0|

### Part B — Top-P (default temperature)

Run the same prompt with 4 different top_p values.

In [ ]:
# Run with different top_p values (temperature stays at model default)
top_p_values = [0.5, 0.75, 0.92, 1.0]

for tp in top_p_values:
    print(f"=== Top-P: {tp} ===\n")

    result = call_mistral("mistral-small-latest", prompt, top_p=tp)
    # result = call_gemini("gemini-2.5-flash", prompt, top_p=tp)

    print(result)
    print("\n" + "=" * 60 + "\n")

=== Top-P: 0.5 ===

Роланд бросился на рельсы, вытолкнув Одетту из-под поезда в последний момент. Взрыв боли пронзил его тело, когда сталь разорвала плоть, но он успел улыбнуться сквозь кровь — она была в безопасности. Одетту обняла его, рыдая, пока его дыхание угасало, а мир вокруг них растворялся в тумане. Он умер героем, спасшим её, и это было всё, что имело значение.


=== Top-P: 0.75 ===

Роланд бросился к Одетте, когда поезд уже набирал скорость, и в последний миг вытолкнул её из опасной зоны. Взрыв обрушился на него, поглотив его тело в ослепительной вспышке. Одетта упала на платформу, дрожа от страха и облегчения, но её глаза наполнились слезами, когда она увидела, как останки Роланда растаяли в воздухе, растворившись в тёмной пустоте. Она знала, что он пожертвовал собой, чтобы спасти её, и эта мысль стала для неё самым тяжёлым бременем и самой светлой надеждой.


=== Top-P: 0.92 ===

**Финал с условием:**

Когда поезд метро с ревом ворвался в туннель, Роланд бросился к Одетте,

**TODO:** Fill in the table:

| Top-P | Creativity (1-5) | Coherence (1-5) | Faithfulness to original (1-5) |
|-------|-----------------|-----------------|-------------------------------|
| 0.5 |3|5|0|
| 0.75 |3|3|0|
| 0.92 |4|3|0|
| 1.0 |5|3|0|

### Conclusion

**TODO:** Write 3-4 sentences answering:
- How does temperature affect the output compared to top_p?
- At what point did quality start degrading for each parameter?
- Which single setting would you recommend for this task and why?

*Your answer:*
* Temperature breaks the entire distribution at once — hence the catastrophic collapse at 1.5. Top_p only trims the token pool, so it degrades smoothly and predictably.
* Temperature started slipping at 1.0, collapsed at 1.5. Top_p held throughout — even 1.0 produced readable text.
* Temperature 0.6 delivered the best results: vivid, emotional, coherent. Top_p never came close.

---
# Task 2: Prompt Engineering — Support Ticket Extraction (3 scores)

You will extract structured data from customer support chat transcripts. The extraction rules have **specific definitions** that may not match common sense — getting them right requires careful prompt engineering.

### Extraction Rules

- **`customer_name`**: The customer's name (string)
- **`product_name`**: The product being discussed (string)
- **`issue_type`**: One of: `billing`, `technical`, `feature_request`, `complaint`
- **`affected_feature`**: The specific feature that's broken or discussed, or `null` if not applicable
- **`workaround_exists`**: `true` ONLY if the support **agent confirmed** a workaround. If the customer says "I tried X and it worked", that does NOT count — it's unverified by the agent.
- **`sentiment_shift`**: Compare ONLY the customer's **first and last messages**. One of: `improved`, `worsened`, `stable`
- **`requires_followup`**: `true` ONLY if the agent explicitly committed to following up (e.g., "I'll get back to you"). Phrases like "contact us if it happens again" do NOT count — that puts the burden on the customer.
- **`promises_made`**: Count of concrete commitments by the agent. Rules:
  - A promise requires a **specific action verb**: "I'll file a report", "I'll send you an email" — each counts as 1
  - "We'll look into it" / "We'll investigate" — does NOT count (too vague)
  - "I'll check with engineering and update the ticket" — counts as **2** (two distinct actions)
  - Repeating the same promise — does NOT count again
- **`resolution_provided`**: `true` ONLY if the agent actually **resolved** the issue during the conversation, not just promised to work on it

In [9]:
# Transcript for Part A
transcript_a = """
Tom: Hey, this is Tom from the finance team. Lisa asked me to reach out — she's been having issues with the reporting module in AnalyticsHub Pro. The revenue charts have been wrong for about a week now.
Agent: Hi Tom! Sorry to hear that. Can you clarify — are you the account holder or is Lisa?
Tom: Lisa Peters is our head of finance and the account owner. I'm just the one who actually uses the tool day to day, so she asked me to sort it out.
Agent: Got it. So the revenue charts in the reporting module are showing incorrect data — is that right?
Tom: Yeah, and it's not just the charts. I noticed the exported PDF reports also have wrong totals. But honestly the charts are the bigger deal because Lisa presents those to the board.
Agent: I see. I've been looking into this and we actually had a data pipeline issue last Tuesday that affected some Enterprise accounts. Let me check... yes, your account was impacted. The good news is we pushed a fix yesterday.
Tom: Oh really? Because I checked this morning and the numbers still looked off.
Agent: Hmm, the fix might take up to 24 hours to fully propagate. Could you try doing a hard refresh on the dashboard? That sometimes forces the latest data to load.
Tom: Sure, let me try... OK I did a hard refresh and the charts look correct now actually. But I'm not sure about the PDF exports.
Agent: Good to hear the charts are working! For the PDFs, I'll create a ticket for our export team to verify those are pulling from the updated pipeline. I'll also flag this to our QA team so they can run a full validation on your account data.
Tom: Alright. And one more thing — we've been asking for months about getting Excel export instead of just PDF. Any update on that?
Agent: I don't have an update on that right now, but I'll check with the product team and circle back. In the meantime, I've gone ahead and extended your trial of the Advanced Analytics add-on by 30 days as a goodwill gesture for the trouble.
Tom: OK, fair enough. Hopefully the PDF thing gets sorted quickly though, we have month-end reporting coming up.
"""

# Expected output:
expected_a = {
    "customer_name": "Lisa Peters",          # Lisa is the account owner. Tom is just reaching out on her behalf — "Lisa asked me to reach out", "she's the account owner"
    "product_name": "AnalyticsHub Pro",
    "issue_type": "technical",               # Data pipeline issue causing wrong numbers — technical, not a complaint
    "affected_feature": "reporting module",   # Both charts and PDF exports are affected, but they're part of the reporting module
    "workaround_exists": True,               # Agent suggested hard refresh, Tom confirmed charts work now, agent said "Good to hear the charts are working!" — agent confirmed the workaround
    "sentiment_shift": "stable",             # First msg: neutral/factual ("having issues"). Last msg: neutral/concerned ("hopefully gets sorted") — both neutral
    "requires_followup": True,               # "I'll check with the product team and circle back" — explicit commitment to get back to them
    "promises_made": 3,                      # 1) create ticket for export team to verify PDFs 2) flag to QA team for full validation 3) check with product team and circle back about Excel export. Extended trial = already done, counts as action but not a promise. "I'll also flag" is separate from the ticket.
    "resolution_provided": False,            # Charts work after refresh but PDFs are still broken. Partial fix ≠ resolution. Extended trial is goodwill, not resolution.
}

### Baseline vs Your Prompt

Below is a **baseline prompt** that simply states the extraction rules. Run it and see which fields it gets wrong.

Then write **your own prompt** using any technique from the seminar — Chain-of-Thought, Few-Shot, Self-Consistency, Chain of Events, or any combination. Your goal: get all fields to match `expected_a`.

In [18]:
# Baseline prompt — just the rules copied from the markdown above
baseline_prompt = f"""Extract the following fields from this support transcript as JSON:
- customer_name: The customer's name
- product_name: The product being discussed
- issue_type: One of: billing, technical, feature_request, complaint
- affected_feature: The specific feature that's discussed, or null
- workaround_exists: true ONLY if the support agent confirmed a workaround. If the customer says "I tried X and it worked", that does NOT count.
- sentiment_shift: Compare ONLY the customer's first and last messages. One of: worsened, stable
- requires_followup: true ONLY if the agent explicitly committed to following up (e.g., "I'll check with the product team and circle back"). "Contact us if it happens again" does NOT count.
- promises_made: Count of concrete commitments by the agent. A promise requires a specific action verb. "We'll look into it" does NOT count. "I'll check with engineering and update the ticket" counts as 2. Repeating the same promise does NOT count again.
- resolution_provided: true ONLY if the agent actually resolved the issue and promise to fix pdf service.

Transcript:
{transcript_a}
"""

baseline_result = call_mistral("mistral-small-latest", baseline_prompt)
#baseline_result = call_gemini("gemini-2.5-flash", baseline_prompt)
print("=== Baseline Result ===\n")
print(baseline_result)
print()
compare_json(baseline_result, expected_a)

=== Baseline Result ===

```json
{
  "customer_name": "Lisa Peters",
  "product_name": "AnalyticsHub Pro",
  "issue_type": "technical",
  "affected_feature": "reporting module",
  "workaround_exists": true,
  "sentiment_shift": "stable",
  "requires_followup": true,
  "promises_made": 3,
  "resolution_provided": false
}
```

=== Comparison Results ===

Correct: 9/9
  ✅ customer_name: Lisa Peters
  ✅ product_name: AnalyticsHub Pro
  ✅ issue_type: technical
  ✅ affected_feature: reporting module
  ✅ workaround_exists: True
  ✅ sentiment_shift: stable
  ✅ requires_followup: True
  ✅ promises_made: 3
  ✅ resolution_provided: False


In [21]:
# TODO: Write your improved prompt using any technique(s) from the seminar
# (Chain-of-Thought, Few-Shot, Self-Consistency, Chain of Events, or a combination)
# Your goal: get ALL fields to match expected_a

my_prompt = f"""Read the support transcript carefully, then extract the fields below as a JSON object.
Before outputting, reason through each field where the rule is non-obvious.

  customer_name:      // Name of the account owner
  product_name:       // Product discussed
  issue_type:         // billing | technical | feature_request | complaint
  affected_feature:   // Feature name or null
  workaround_exists:  // true = agent confirmed workaround. Customer workarounds don't count.
  sentiment_shift:    // First vs last customer message only:  negative | stable
  requires_followup:  // true = agent promised to follow up. "Contact us if…" = false.
  promises_made:      // Count of distinct, specific agent promises. Vague = 0. Duplicates = 1.
  resolution_provided:// true = issue resolved + agent promised to fix PDF service.

Transcript:
{transcript_a}
"""

my_result = call_mistral("mistral-small-latest", my_prompt)
# my_result = call_gemini("gemini-2.5-flash", my_prompt)
print("=== Your Result ===\n")
print(my_result)
print()
compare_json(my_result, expected_a)

=== Your Result ===

```json
{
  "customer_name": "Lisa Peters",
  "product_name": "AnalyticsHub Pro",
  "issue_type": "technical",
  "affected_feature": "reporting module",
  "workaround_exists": true,
  "sentiment_shift": "stable",
  "requires_followup": true,
  "promises_made": 3,
  "resolution_provided": false
}
```

### Reasoning:

1. **customer_name**:
   - The transcript mentions "Lisa Peters is our head of finance and the account owner."
   - Even though Tom is the one speaking, the account owner is Lisa Peters, so we use her name.

2. **product_name**:
   - The product discussed is "AnalyticsHub Pro" as mentioned by Tom.

3. **issue_type**:
   - The issue is about incorrect data in the reporting module and PDF exports, which is a technical issue.

4. **affected_feature**:
   - The affected feature is the "reporting module" as explicitly mentioned in the transcript.

5. **workaround_exists**:
   - The agent suggests a hard refresh as a workaround, which is confirmed to work ("t

---
# Task 3: Structured Output with Pydantic — Job Posting Extraction (2 scores)

Define nested Pydantic models to represent a job posting, then use `parse_mistral` (or another provider) to extract structured data from a real-looking job posting.

### Your task:
1. Define the Pydantic models (schema is provided below — fill in the fields)
2. Write a prompt that instructs the LLM to extract data according to the schema
3. Call `parse_mistral` / `parse_gemini` / `parse_openrouter` to get a parsed result
4. Compare your result against the expected output

### Extraction rules:
- `salary_min` / `salary_max`: `null` if salary is "competitive" or not mentioned
- `min_years_experience`: extract the MINIMUM from a range ("3-5 years" → 3)
- `remote_policy`: one of `remote`, `hybrid`, `onsite`
- `required_skills` vs `preferred_skills`: "must have" / "required" → required; "nice to have" / "a plus" / "bonus" → preferred
- `has_equity`: only `true` if explicitly mentioned for this role (not "equity for senior roles" when this is a mid-level role)

In [22]:
job_posting = """
Backend Engineer — FinFlow (Series B Fintech Startup)

About us: FinFlow is a fast-growing fintech startup (120 employees) building the next generation
of payment infrastructure. Based in Berlin, Germany.

Role: We're looking for a mid-level Backend Engineer to join our Payments team.
You'll design and maintain APIs that process millions of transactions daily.

Requirements:
- 3-5 years of experience in backend development
- Strong proficiency in Python and SQL (required)
- Experience with PostgreSQL and Redis (required)
- Familiarity with event-driven architectures
- Bachelor's degree in CS or equivalent practical experience

Nice to have:
- Experience with Kubernetes and AWS
- Knowledge of payment protocols (ISO 8583, PCI-DSS)
- Contributions to open-source projects

What we offer:
- Competitive salary (we benchmark against top-tier companies)
- Equity package for senior roles
- Remote-friendly — we have a Berlin office but you can work from anywhere in EU.
  We do quarterly team offsites (3 days each).
- 30 days PTO, learning budget, gym membership

Apply at careers@finflow.io
"""

In [77]:
class Company(BaseModel):
    name: str
    industry: str
    size: Literal["startup", "enterprise"]
    location: str

class Compensation(BaseModel):
    salary_min: Optional[int] = Field(None, description="null if salary is 'competitive' or not mentioned")
    salary_max: Optional[int] = Field(None, description="null if salary is 'competitive' or not mentioned")
    currency: Optional[str] = Field(None, description="null if not explicitly mentioned")
    has_equity: bool = Field(False, description="True ONLY if explicitly mentioned for THIS specific role")
    benefits: List[str] = []

class Requirements(BaseModel):
    min_years_experience: Optional[int] = Field(None, description="Minimum from a range, e.g., '3-5 years' -> 3")
    education_level: Optional[Literal["bachelor", "master", "phd", "high school"]] = None
    required_skills: List[str] = Field(..., description="'must have' or 'required' skills")
    preferred_skills: List[str] = Field(..., description="'nice to have', 'a plus', or 'bonus' skills")
    languages: List[str] = []

class RoleDetails(BaseModel):
    title: str
    seniority: Literal["intern", "junior", "mid", "senior", "lead"]
    department: str
    remote_policy: Literal["hybrid", "onsite"]
    responsibilities: List[str] = []

class JobPosting(BaseModel):
    company: Company
    compensation: Compensation
    requirements: Requirements
    role: RoleDetails

In [78]:
# TODO: Use parse_mistral (or parse_gemini) to extract structured data

#result = parse_mistral("mistral-small-latest", job_posting, JobPosting)
result = parse_gemini("gemini-2.5-flash", job_posting, JobPosting)

print(result)

company=Company(name='FinFlow', industry='Fintech', size='startup', location='Berlin, Germany') compensation=Compensation(salary_min=None, salary_max=None, currency=None, has_equity=False, benefits=['30 days PTO', 'learning budget', 'gym membership']) requirements=Requirements(min_years_experience=3, education_level='bachelor', required_skills=['Python', 'SQL', 'PostgreSQL', 'Redis'], preferred_skills=['event-driven architectures', 'Kubernetes', 'AWS', 'ISO 8583', 'PCI-DSS', 'contributions to open-source projects'], languages=[]) role=RoleDetails(title='Backend Engineer', seniority='mid', department='Payments', remote_policy='hybrid', responsibilities=['design and maintain APIs that process millions of transactions daily'])


In [79]:
# Verify your result
expected_posting = {
    "company": {"name": "FinFlow", "industry": "fintech", "size": "startup", "location": "Berlin, Germany"},
    "compensation": {
        "salary_min": None,
        "salary_max": None,
        "currency": None,
        "has_equity": False,
    },
    "requirements": {
        "min_years_experience": 3,
        "education_level": "bachelor",
    },
    "role": {
        "title": "Backend Engineer",
        "seniority": "mid",
        "department": "Payments",
        "remote_policy": "hybrid",
    },
}

# Compare — skip list fields (order/wording may vary)
compare_json(
    result.model_dump() if hasattr(result, 'model_dump') else result,
    expected_posting,
    skip_fields=["required_skills", "preferred_skills", "languages", "benefits", "responsibilities"]
)

=== Comparison Results ===

Correct: 14/14
  ✅ company.name: FinFlow
  ✅ company.industry: fintech
  ✅ company.size: startup
  ✅ company.location: Berlin, Germany
  ✅ compensation.salary_min: None
  ✅ compensation.salary_max: None
  ✅ compensation.currency: None
  ✅ compensation.has_equity: False
  ✅ requirements.min_years_experience: 3
  ✅ requirements.education_level: bachelor
  ✅ role.title: Backend Engineer
  ✅ role.seniority: mid
  ✅ role.department: Payments
  ✅ role.remote_policy: hybrid


---
# Task 4: Schema-Guided Reasoning — Travel Itinerary Planner (3 scores)

Build a travel planner using SGR. The user provides minimal input (destination, number of days, travel type, a few must-visit places). The LLM uses its own knowledge to fill in attractions, build a day-by-day plan, and validate it.

You will implement all three SGR patterns:
1. **Cascade**: Parse request → Generate attractions → Build daily plan → Validate
2. **Router**: Different planning logic for adventure vs relaxation vs cultural trips
3. **Cycle**: Refine the plan until all constraints are met

### 4.1 Cascade Pattern

Build a schema where each step is a **field** of a single class. The LLM fills them in order — when generating Step 2, it already sees Step 1's output. **One call, one schema.**

Define a `TravelPlanCascade` class with three steps:

**Step 1 — `request: TravelRequest`**: Parse the user's free-text request.
- `destination`: str, `num_days`: int
- `travel_type`: str — "adventure", "relaxation", or "cultural"
- `must_visit`: List[str]

**Step 2 — `discovery: DestinationInfo`**: Generate knowledge about the destination.
- `destination`: str
- `attractions`: List[Attraction] — each has `name`, `category`, `typical_duration_hours`, `description`
- `best_area_to_stay`: str, `local_tips`: List[str]

**Step 3 — `itinerary: Itinerary`**: Build a day-by-day plan using the discovered attractions.
- `destination`: str
- `days`: List[DayPlan] — each has `day_number`, `morning` (Activity), `afternoon` (Activity), `evening` (Activity), `total_active_hours`

Helper models needed: `Attraction` (name, category, typical_duration_hours, description), `Activity` (name, duration_hours, category), `DayPlan` (day_number, morning, afternoon, evening, total_active_hours).

The final schema:
```python
class TravelPlanCascade(BaseModel):
    request: TravelRequest
    discovery: DestinationInfo
    itinerary: Itinerary
```

One single call to `parse_mistral` with `TravelPlanCascade`.

In [ ]:
# TODO: Define all Pydantic schemas for the cascade (Attraction, Activity, DayPlan, and the 4 step schemas)

# YOUR SCHEMAS HERE


In [ ]:
# Test input
user_request = "I want a 3-day cultural trip to Rome. I must visit the Colosseum and the Vatican."

# TODO: Make ONE call with TravelPlanCascade as the response format

# result = parse_mistral("mistral-small-latest", user_request, TravelPlanCascade)
# result = parse_gemini("gemini-2.5-flash", user_request, TravelPlanCascade)

# Pretty-print the result:
# pretty_print(result)

### 4.2 Router Pattern

Now **extend** the cascade with type-specific requirements. Different travel types have different constraints — the router adds a requirements layer that depends on the travel type.

The idea: the same cascade (request → discovery → itinerary) but now with a `requirements` field that uses `Union` to pick the right requirements schema based on the travel type.

**`AdventureRequirements`** (kind: Literal["adventure"]):
- `physical_intensity_per_day`: int (1-5), `outdoor_ratio`: float (0.0-1.0)
- `equipment_needed`: List[str], `best_season`: str

**`RelaxationRequirements`** (kind: Literal["relaxation"]):
- `max_activities_per_day`: int (1-2 for relaxation)
- `downtime_hours_per_day`: float, `spa_or_wellness`: bool

**`CulturalRequirements`** (kind: Literal["cultural"]):
- `museums_count`: int, `historical_sites_count`: int
- `local_experiences`: List[str]

**`TravelPlanWithRequirements`**: extends the cascade with routing
```python
class TravelPlanWithRequirements(BaseModel):
    request: TravelRequest
    requirements: Union[AdventureRequirements, RelaxationRequirements, CulturalRequirements]
    discovery: DestinationInfo
    itinerary: Itinerary
```

The model first parses the request, then picks the right requirements schema (routing), then discovers and plans accordingly — **all in one call**.

Test with three requests (expected routing: adventure, relaxation, cultural).

In [ ]:
# TODO: Define the requirement schemas and TravelPlanWithRequirements
# Reuse TravelRequest, DestinationInfo, Itinerary from 4.1

# YOUR SCHEMAS HERE


In [ ]:
# Test the router with 3 requests
test_requests = [
    "Plan a 4-day hiking and kayaking trip to Patagonia.",
    "I need a 5-day spa and beach retreat in Bali. Keep it very chill.",
    "3 days exploring museums and history in Athens.",
]

# Expected routing: adventure, relaxation, cultural

for request in test_requests:
    # result = parse_mistral("mistral-small-latest", request, TravelPlanWithRequirements)
    # result = parse_gemini("gemini-2.5-flash", request, TravelPlanCascade)
    # print(f"Request: {request[:60]}...")
    # print(f"  Routed to: {result.requirements.kind}")
    # pretty_print(result)
    # print()
    pass

### 4.3 Cycle Pattern

Now **extend** the schema with validation and iterative refinement. The model generates a plan, validates it against constraints, finds issues, and refines — **all within one structured output** containing a list of iterations.

This builds on top of what you created in 4.1 and 4.2 — now adding a constraint-checking and refinement loop.

**Constraints to enforce:**
- All must-visit places are included
- No day exceeds 10 hours of active time
- No two physically exhausting activities back-to-back
- Activities on the same day should be geographically reasonable

**`ConstraintCheck`**:
- `all_must_visit_included`: bool, `missing_places`: List[str]
- `any_day_over_10_hours`: bool, `over_limit_days`: List[int]
- `back_to_back_exhausting`: bool, `exhausting_pairs`: List[str]
- `geographically_reasonable`: bool, `geographic_issues`: List[str]
- `all_constraints_passed`: bool

**`PlanIteration`**: one cycle of propose → validate → identify fixes
- `proposed_plan`: List[DayPlan]
- `constraint_check`: ConstraintCheck
- `issues_found`: List[str], `fixes_to_apply`: List[str]
- `iteration_quality`: int (1-10)

**`TravelPlanWithCycle`**: the full schema — extends previous work with validation
```python
class TravelPlanWithCycle(BaseModel):
    request: TravelRequest
    requirements: Union[AdventureRequirements, RelaxationRequirements, CulturalRequirements]
    discovery: DestinationInfo
    iterations: List[PlanIteration]  # the cycle — multiple refinement rounds
    final_plan: List[DayPlan]
    all_constraints_met: bool
```

One call — the model fills request → requirements → discovery → then iterates through multiple plan refinements, each time checking constraints and improving.

In [ ]:
# TODO: Define ConstraintCheck, PlanIteration, and TravelPlanWithCycle
# Reuse schemas from 4.1 and 4.2 — extend them with the validation layer

# YOUR SCHEMAS HERE


In [ ]:
# Test input for the cycle
cycle_request = """
Plan a 3-day trip to Barcelona. Travel type: cultural + food.
Must visit: Sagrada Familia, Park Guell, La Boqueria market, Gothic Quarter.

Constraints:
- No day should have more than 10 hours of activities
- Don't put two long walking tours back-to-back
- Keep activities on the same day in nearby neighborhoods
- All 4 must-visit places must be included

Generate 3 iterations, improving the plan each time.
"""

# TODO: Make ONE call with TravelPlanWithCycle

# result = parse_mistral("mistral-small-latest", cycle_request, TravelPlanWithCycle)
# parse_gemini("gemini-2.5-flash", cycle_request, TravelPlanWithCycle)

# Pretty-print the result:
# pretty_print(result)

---
# Task 5: Design Your Own SGR — Group Trip Planner (3 scores)

A group of 4 friends with **different preferences** is traveling together for 5 days. Design your own SGR schema that creates a plan where everyone is happy — **in a single LLM call**.

### The group:
- **Alex**: Wants adventure — hiking, water sports, anything outdoors
- **Sam**: Loves culture — museums, historical sites, local traditions  
- **Jordan**: All about food — cooking classes, food tours, local restaurants
- **Riley**: Needs relaxation — beaches, spas, scenic views, no rushing

### Destination: Belgrade, Serbia (5 days)
*(Feel free to change the destination to any city you'd like — just update the `destination` variable below)*

### Requirements:
- Each person must have **at least 2 activities** matching their primary interest
- Each day must have **at least 1 group activity** that everyone can enjoy together
- No day should exceed 10 hours of active time
- The plan should feel balanced — no one person's interests should dominate

### Your task:
Design a **single top-level Pydantic class** (e.g. `GroupTripPlan`) that guides the LLM through all the reasoning steps via its nested structure. Remember: SGR means **one call, one schema** — the structure itself controls the reasoning.

The LLM call and result printing are already written below — you just need to define the schema.

**Hint**: Think about what reasoning steps would help — maybe first analyze each person's needs, then find activities in the city, then identify group-friendly activities, then build daily plans, then validate satisfaction? Each step should be a nested field in your top-level class.

In [ ]:
# TODO: Define your Pydantic schemas for the group trip planner.
# Your top-level class (e.g. GroupTripPlan) should contain nested classes
# that guide the LLM through reasoning steps.
#
# Example structure (you can change this completely):
# - Step 1: Analyze each person's preferences
# - Step 2: Find activities in Lisbon that match each person
# - Step 3: Identify group-friendly activities
# - Step 4: Build daily plans with a balance of individual + group activities
# - Step 5: Validate that each person has enough matching activities
#
# IMPORTANT: The top-level class must be called GroupTripPlan
# (it's used in the execution cell below)

# YOUR SCHEMAS HERE


In [ ]:
# Execute the SGR — one call with your GroupTripPlan schema

destination = "Belgrade, Serbia"  # <-- change to any city you want!

user_request = f"""
Plan a 5-day trip to {destination} for a group of 4 friends:
- Alex: wants adventure (hiking, water sports, outdoors)
- Sam: loves culture (museums, historical sites, local traditions)
- Jordan: all about food (cooking classes, food tours, local restaurants)
- Riley: needs relaxation (beaches, spas, scenic views, no rushing)

Requirements:
- Each person must have at least 2 activities matching their interest
- Each day must have at least 1 group activity everyone can enjoy
- No day should exceed 10 hours of active time
- The plan should be balanced — no one person's interests should dominate
"""

result = parse_mistral("mistral-small-latest", user_request, GroupTripPlan)
# result = parse_gemini("gemini-2.5-flash", user_request, GroupTripPlan)

print(result.model_dump_json(indent=2))

In [ ]:
# Pretty-print the result and verify
pretty_print(result)

print("\n=== Verification ===\n")
print("Check the output above and verify:")
print("1. Does each person (Alex, Sam, Jordan, Riley) have at least 2 matching activities?")
print("2. Does each day have at least 1 group activity?")
print("3. Is any day over 10 hours of active time?")
print("4. Does the plan feel balanced across all interests?")
print()
print("If your schema includes a validation step, those checks should be visible above.")